# 07 · O GPT completo no Shakespeare

Todas as peças estão prontas. Agora o objetivo muda: em vez de comparar peças num modelo minúsculo, treinar um GPT **de verdade** (alguns milhões de parâmetros) e ver o que a escala traz, e o problema que ela cria com só 1 MB de texto: **overfitting**.

Duas novidades pequenas:
- **atenção em lote** (`CausalSelfAttention`): a mesma conta das várias cabeças, com todas num tensor só. Bem mais rápida na GPU.
- **dropout**: durante o treino, zera ao acaso uma fração das ativações, para o modelo não depender de detalhes específicos do texto de treino.

Diário: `F1-SHK`.

In [ ]:
%load_ext autoreload
%autoreload 2

import inspect
import time

import matplotlib.pyplot as plt
import torch

from lab.data import get_batch, shakespeare
from lab.model import GPT, CausalSelfAttention, GPTConfig, MultiHeadAttention, count_params
from lab.plot import compare, plot_attention, table
from lab.train import generate_text, load_checkpoint, pick_device, train

DEVICE = pick_device()
ds = shakespeare()
tok, V = ds.tok, ds.tok.vocab_size
print(ds, "| dispositivo:", DEVICE)


def sync():
    """Espera a GPU terminar (as operações são assíncronas), para medir tempo direito."""
    if DEVICE == "mps":
        torch.mps.synchronize()
    elif DEVICE == "cuda":
        torch.cuda.synchronize()

## O modelo inteiro
Leia com calma: cada linha corresponde a algo que você construiu nos notebooks 01 a 06.

In [ ]:
print(inspect.getsource(GPT))

## Atenção em lote: mesma conta, mais rápida

`CausalSelfAttention` calcula q, k e v de todas as cabeças com **uma** multiplicação (`qkv`) e reorganiza o tensor para `(B, n_head, T, head_size)`. Para provar que é a mesma conta, copie os pesos da versão com cabeças separadas e compare as saídas.

In [ ]:
torch.manual_seed(0)
d, nh, T = 64, 4, 32
mha, csa = MultiHeadAttention(d, nh, T), CausalSelfAttention(d, nh, T)
with torch.no_grad():
    csa.qkv.weight.copy_(torch.cat([
        torch.cat([h.query.weight for h in mha.heads]),
        torch.cat([h.key.weight for h in mha.heads]),
        torch.cat([h.value.weight for h in mha.heads]),
    ]))
    csa.proj.load_state_dict(mha.proj.state_dict())
x = torch.randn(2, T, d)
print("mesma saída:", torch.allclose(mha(x), csa(x), atol=1e-5))
print("mesmos parâmetros:", count_params(mha), count_params(csa))

**Minha previsão:** _(quantas vezes mais rápida é a versão em lote num bloco de tamanho real, 6 cabeças, d_model 384, contexto 256?)_

In [ ]:
def time_fwd_bwd(layer, iters=10):
    x = torch.randn(16, 256, 384, device=DEVICE, requires_grad=True)
    layer(x).sum().backward()  # aquecimento
    sync()
    t0 = time.time()
    for _ in range(iters):
        layer(x).sum().backward()
    sync()
    return (time.time() - t0) / iters * 1000

for Attn in [MultiHeadAttention, CausalSelfAttention]:
    torch.manual_seed(0)
    print(f"{Attn.__name__:20} {time_fwd_bwd(Attn(384, 6, 256).to(DEVICE)):6.1f} ms por passo")

## Um GPT médio (~5M parâmetros)

6 blocos, 8 cabeças, `d_model = 256`, contexto de 128 caracteres. É 6× maior que o modelo da Fase 0.

In [ ]:
def medio(dropout=0.0):
    torch.manual_seed(1337)
    return GPT(GPTConfig(vocab_size=V, block_size=128, d_model=256, n_layer=6, n_head=8, attn="batched", dropout=dropout))

TRAIN = dict(steps=3000, lr=6e-4, batch_size=32, grad_clip=1.0, eval_every=250, device=DEVICE)
print(f"{count_params(medio()):,} parâmetros")

### Quanto vai demorar?
Antes de um treino longo, meça alguns passos e extrapole.

In [ ]:
m = medio().to(DEVICE)
opt = torch.optim.AdamW(m.parameters(), lr=6e-4)
for i in range(25):
    if i == 5:  # os primeiros passos incluem aquecimento
        sync()
        t0 = time.time()
    _, loss = m(*get_batch(ds.train, TRAIN["batch_size"], 128, DEVICE))
    opt.zero_grad()
    loss.backward()
    opt.step()
sync()
per_step = (time.time() - t0) / 20
print(f"{per_step * 1000:.0f} ms por passo -> {TRAIN['steps']} passos em ~{per_step * TRAIN['steps'] / 60:.0f} min (sem contar avaliações)")
del m, opt

## Treino de referência, sem dropout

**Minha previsão:** _(a loss de validação passa de 1,5? em algum momento ela para de cair enquanto a de treino continua?)_

In [ ]:
run = train(medio(), ds, name="07-medio", save=True, **TRAIN)
print(run.sample)

---
## Experimentos

### 1. Dropout contra o overfitting
**Minha previsão:** _(com dropout 0,2, a loss de treino fica maior ou menor? e a de validação? a distância entre as duas?)_

In [ ]:
run_drop = train(medio(dropout=0.2), ds, name="07-medio-drop0.2", save=True, **TRAIN)
compare(["07-medio", "07-medio-drop0.2"], title="GPT médio: sem e com dropout")
table(["06-norm-pre", "07-medio", "07-medio-drop0.2"])

### 2. Gerar texto: temperatura e top-k
Carregue o melhor modelo salvo e gere com configurações diferentes. `top_k` descarta tudo fora dos k caracteres mais prováveis antes de sortear.

**Minha previsão:** _(qual combinação dá o texto mais "Shakespeare"? onde ele começa a repetir?)_

In [ ]:
runs = {"07-medio": run, "07-medio-drop0.2": run_drop}
best = min(runs, key=lambda n: runs[n].final["val_loss"])
model, tok = load_checkpoint(best, DEVICE)
print("modelo:", best)
for temp, top_k in [(1.0, None), (0.8, None), (0.8, 10), (0.5, 5)]:
    print(f"\n--- temperatura {temp}, top_k {top_k} ---")
    print(generate_text(model, tok, "ROMEO:\n", 300, temperature=temp, top_k=top_k))

### 3. O que as cabeças olham num modelo treinado
Primeiro e último bloco, todas as cabeças.

In [ ]:
text = "ROMEO:\nBut, soft! what light through yonder window breaks?"[:48]
idx = torch.tensor([tok.encode(text)], device=DEVICE)
model(idx)
for layer in [0, len(model.blocks) - 1]:
    att = model.blocks[layer].attn.last_att[0].cpu()  # (n_head, T, T)
    fig, axs = plt.subplots(2, 4, figsize=(20, 10))
    for h, ax in enumerate(axs.flat):
        plot_attention(att[h], list(text), ax=ax, title=f"bloco {layer}, cabeça {h}")
    plt.tight_layout()
    plt.show()

### 4. (Opcional, longo) O GPT de ~10M parâmetros
A configuração final do vídeo do Karpathy: 6 blocos, 6 cabeças, `d_model = 384`, contexto de 256, dropout 0,2. No MacBook Air deve levar perto de uma hora (use a célula "quanto vai demorar?" com essa configuração para estimar). Melhor rodar no terminal, com o notebook livre:

```bash
python -m lab.train --name 07-gpt10m --dataset shakespeare --n_layer 6 --n_head 6 --d_model 384 \
    --block_size 256 --dropout 0.2 --attn batched --steps 5000 --lr 3e-4 --batch_size 64 \
    --grad_clip 1.0 --eval_every 250 --save
```

Depois, compare aqui: `compare(["07-medio-drop0.2", "07-gpt10m"])` e gere texto com `load_checkpoint("07-gpt10m")`.

In [ ]:
compare(["07-medio-drop0.2", "07-gpt10m"], title="Médio vs. 10M")
table(["07-medio-drop0.2", "07-gpt10m"])

---
## Para o diário (`F1-SHK-01`)

- Onde a loss de validação parou sem dropout, e quando começou o overfitting? O dropout resolveu?
- O texto gerado tem palavras reais? Nomes de personagens? Estrutura de peça (falas, quebras de linha)?
- Algum padrão de atenção reconhecível no último bloco (início de palavra, quebra de linha, mesmo caractere)?

**Próximo:** [08 · TinyStories](08_tinystories.ipynb): mais dados, para o modelo parar de decorar.